<a href="https://colab.research.google.com/github/zuelmaruixin/AndroidReader/blob/main/6000_Q_A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall bitsandbytes -y
!pip install -U "bitsandbytes>=0.46.1"
!pip install -U transformers datasets peft trl accelerate



import gc
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen3-1.7B"

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available(),
assert torch.cuda.is_bf16_supported(),
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

def extract_first_turn(example):
    text = example["text"]

    try:
        if "### Human:" not in text or "### Assistant:" not in text:
            return {"text": ""}

        text = text.strip()

        human_start = text.find("### Human:")
        assistant_start = text.find("### Assistant:", human_start)

        if human_start == -1 or assistant_start == -1:
            return {"text": ""}

        human_part = text[human_start + len("### Human:"):assistant_start].strip()

        next_human_start = text.find("### Human:", assistant_start + len("### Assistant:"))
        if next_human_start == -1:
            assistant_part = text[assistant_start + len("### Assistant:"):].strip()
        else:
            assistant_part = text[assistant_start + len("### Assistant:"):next_human_start].strip()

        if not human_part or not assistant_part:
            return {"text": ""}

        bad_markers = ["### Human:", "### Assistant:"]
        if any(marker in human_part for marker in bad_markers):
            return {"text": ""}
        if any(marker in assistant_part for marker in bad_markers):
            return {"text": ""}

        backward_text = (
            f"以下是一段 AI 的回答:\n{assistant_part}\n\n"
            f"请推断出引发上述回答的用户指令(Instruction):\n{human_part}"
        )
        return {"text": backward_text}

    except Exception:
        return {"text": ""}

print("构建反向数据集...")
backward_dataset = dataset.map(extract_first_turn).filter(lambda x: len(x["text"]) > 0)

split_ds = backward_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_ds["train"]
eval_data = split_ds["test"]

print("train size:", len(train_data))
print("eval size:", len(eval_data))
print("\n样例：\n")
print(train_data[0]["text"][:1000])
try:
    del model, trainer
except:
    pass
gc.collect()
torch.cuda.empty_cache()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.config.use_cache = False
training_args = SFTConfig(
    output_dir="./qwen3_backward_lora",
    dataset_text_field="text",
    max_length=512,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    max_steps=100,
    logging_steps=5,

    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    load_best_model_at_end=True,

    bf16=True,
    fp16=False,
    bf16_full_eval=True,
    fp16_full_eval=False,

    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    processing_class=tokenizer,
)
trainer.train()
save_path = "./qwen3_backward_lora_adapter"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"LoRA adapter 已保存到: {save_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
CUDA available: True
Device: NVIDIA A100-SXM4-80GB
bf16 supported: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/985 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/985 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
20,1.820540,1.641221
40,1.559511,1.509363
60,1.572149,1.481959
80,1.463877,1.465670
100,1.557174,1.463394


LoRA adapter 已保存到: ./qwen3_backward_lora_adapter


In [3]:
from huggingface_hub import login
login()

In [4]:
trainer.model.push_to_hub("xinxinwanshishunli/qwen3-backward-lora")
tokenizer.push_to_hub("xinxinwanshishunli/qwen3-backward-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  10%|9         | 1.25MB / 12.9MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp_48hpdp3/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/xinxinwanshishunli/qwen3-backward-lora/commit/c975ab79658d923c8ca7f4741599643f54faae14', commit_message='Upload tokenizer', commit_description='', oid='c975ab79658d923c8ca7f4741599643f54faae14', pr_url=None, repo_url=RepoUrl('https://huggingface.co/xinxinwanshishunli/qwen3-backward-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='xinxinwanshishunli/qwen3-backward-lora'), pr_revision=None, pr_num=None)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id = "Qwen/Qwen3-1.7B"
adapter_path = "./qwen3_backward_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

test_responses = [
    "The capital of Japan is Tokyo.",
    "Monopsony refers to a market structure in which there is only one buyer in a market.",
    "To improve sleep quality, you should maintain a consistent bedtime, avoid caffeine late in the day, and reduce screen exposure before bed.",
    "Here are three ways to reduce stress: exercise regularly, practice mindfulness, and get enough sleep.",
    "Python is a high-level programming language known for its readability and wide range of applications, including web development, data science, and automation."
]

for i, test_response in enumerate(test_responses, 1):
    prompt = f"""以下是一段 AI 的回答:
{test_response}

请推断出引发上述回答的用户指令(Instruction):
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n===== Example {i} =====")
    print(result)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



===== Example 1 =====
以下是一段 AI 的回答:
The capital of Japan is Tokyo.

请推断出引发上述回答的用户指令(Instruction):
What is the capital of Japan?

===== Example 2 =====
以下是一段 AI 的回答:
Monopsony refers to a market structure in which there is only one buyer in a market.

请推断出引发上述回答的用户指令(Instruction):
What is a monopsony?

===== Example 3 =====
以下是一段 AI 的回答:
To improve sleep quality, you should maintain a consistent bedtime, avoid caffeine late in the day, and reduce screen exposure before bed.

请推断出引发上述回答的用户指令(Instruction):
What are some ways to improve sleep quality?

===== Example 4 =====
以下是一段 AI 的回答:
Here are three ways to reduce stress: exercise regularly, practice mindfulness, and get enough sleep.

请推断出引发上述回答的用户指令(Instruction):
What are three ways to reduce stress?

===== Example 5 =====
以下是一段 AI 的回答:
Python is a high-level programming language known for its readability and wide range of applications, including web development, data science, and automation.

请推断出引发上述回答的用户指令(Instruction):
What is Py

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!cp -r /content/qwen3_backward_lora_adapter /content/drive/MyDrive/

print("文件夹已成功复制到 Google Drive 根目录！")

文件夹已成功复制到 Google Drive 根目录！


In [2]:
# !pip uninstall bitsandbytes -y
# !pip install -U "bitsandbytes>=0.46.1"
# !pip install -U transformers datasets peft trl accelerate
# !pip uninstall -y datasets
# !pip install "datasets<4.0.0"
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset
import random
import pandas as pd
drive.mount('/content/drive')
model_id = "Qwen/Qwen3-1.7B"
adapter_path = "/content/drive/MyDrive/qwen3_backward_lora_adapter"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4"
)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("正在从云盘加载反向模型适配器...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()
print("正在加载并过滤 LIMA 数据集...")
lima = load_dataset("GAIR/lima", split="train",trust_remote_code=True)
single_turn_lima = lima.filter(lambda x: len(x["conversations"]) == 2)

random.seed(42)
indices = random.sample(range(len(single_turn_lima)), 150)
sampled_data = single_turn_lima.select(indices)
generated_data = []
print("开始生成指令...")
for i, ex in enumerate(sampled_data):
    original_instruction = ex["conversations"][0]
    response = ex["conversations"][1]
    prompt = f"以下是一段 AI 的回答:\n{response}\n\n请推断出引发上述回答的用户指令(Instruction):\n"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    generated_instruction = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    generated_data.append({
        "generated_instruction": generated_instruction,
        "response": response,
        "original_instruction": original_instruction
    })

    if (i + 1) % 10 == 0:
        print(f"进度: {i + 1}/150")

generated_data = [
    x for x in generated_data
    if x["generated_instruction"] and len(x["generated_instruction"].strip()) > 3
]

print("\n--- 生成示例展示 ---")
for i in range(5):
    print(f"示例 {i+1}:")
    print(f"生成的指令: {generated_data[i]['generated_instruction']}")
    print(f"原始回答: {generated_data[i]['response'][:100]}...\n")

df = pd.DataFrame(generated_data)
df.to_json("self_augmentation_results.jsonl", orient="records", lines=True)
!cp self_augmentation_results.jsonl /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

正在从云盘加载反向模型适配器...
正在加载并过滤 LIMA 数据集...


plain_text/train/0000.parquet:   0%|          | 0.00/1.68M [00:00<?, ?B/s]

plain_text/test/0000.parquet:   0%|          | 0.00/27.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1030 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1030 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


开始生成指令...
进度: 10/150
进度: 20/150
进度: 30/150
进度: 40/150
进度: 50/150
进度: 60/150
进度: 70/150
进度: 80/150
进度: 90/150
进度: 100/150
进度: 110/150
进度: 120/150
进度: 130/150
进度: 140/150
进度: 150/150

--- 生成示例展示 ---
示例 1:
生成的指令: advice-giver is trying to manipulate you, you can either ignore them or confront them directly. If they are trying to manipulate you, you can say something like, "I don't want to hear anything else from you. I'm going to take my own advice and do it on my own." If they are trying to help you, you can say something like, "Thank you for your advice. I'm going to take it into consideration and do my best to handle this on my own."

## Politely shut down the conversation

1. Politely decline. If you are not interested in the advice, you can politely decline it
原始回答: When someone gives you unsolicited advice, it can be tricky to know how to respond, no matter how we...

示例 2:
生成的指令: What is the difference between using prop() and attr() when enabling or disabling elements in jQuery?
原

In [2]:
import re
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
INPUT_FILE = "self_augmentation_results.jsonl"
SCORED_OUTPUT_FILE = "self_curation_scored_v3.jsonl"
CURATED_OUTPUT_FILE = "self_curation_high_quality_v3.jsonl"

MODEL_ID = "Qwen/Qwen3-1.7B"
df = pd.read_json(INPUT_FILE, lines=True)

if "generated_instruction" in df.columns and "instruction" not in df.columns:
    df["instruction"] = df["generated_instruction"]

keep_cols = [c for c in ["instruction", "response", "original_instruction"] if c in df.columns]
df = df[keep_cols].copy()

print("原始样本数:", len(df))
print(df.head(3))
def basic_filter(row):
    instr = str(row["instruction"]).strip()
    resp = str(row["response"]).strip()

    bad_phrases = [
        "请推断出引发上述回答",
        "Instruction):",
        "### Human:",
        "### Assistant:",
        "## ",
        "</Reason>",
        "</Score>",
        "Reason:",
        "Score:"
    ]

    if not instr or not resp:
        return False
    if len(instr) < 5 or len(resp) < 5:
        return False
    if len(instr) > 300:
        return False
    if len(resp) > 5000:
        return False
    if "\n\n" in instr:
        return False
    if instr.count("\n") > 1:
        return False
    if any(x in instr for x in bad_phrases):
        return False
    return True

df = df[df.apply(basic_filter, axis=1)].reset_index(drop=True)

print("\n清洗后样本数:", len(df))
print(df.head(3))
print("\n加载 Judge 模型（bf16）...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# 关键：decoder-only 模型批量推理必须左填充
tokenizer.padding_side = "left"

judge_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

judge_model.eval()

print("Judge 模型加载完成。")
print("padding_side =", tokenizer.padding_side)
print("bf16 supported =", torch.cuda.is_bf16_supported())

# =========================
# 构造评分 prompt
# 不再让模型自由输出长文本，只让它输出 1~5
# =========================
SYSTEM_PROMPT = """You are an expert evaluator for instruction-response quality.

Evaluate whether the instruction-response pair is a high-quality example for instruction tuning.

Scoring rubric:
1 = Bad. Nonsensical, corrupted, or clearly mismatched.
2 = Poor. Weakly related, unnatural, or incomplete.
3 = Acceptable. Somewhat related, but vague or only partially aligned.
4 = Good. Mostly clear and well-matched, with minor issues.
5 = Excellent. Natural instruction, strong match, and helpful response.

Output only one digit: 1, 2, 3, 4, or 5.
"""

FEW_SHOTS = """Example 1
Instruction: What is the capital of Japan?
Response: The capital of Japan is Tokyo.
Score: 5

Example 2
Instruction: What is the purpose of the | symbol in bash?
Response: You are using | (pipe) to direct the output of a command into another command.
Score: 5

Example 3
Instruction: How do I clean slate?
Response: Slate is a stone that brings natural beauty into the home, and can be expensive to install. Regular maintenance is important.
Score: 2

Example 4
Instruction: advice-giver is trying to manipulate you, you can either ignore them or confront them directly.
Response: When someone gives you unsolicited advice, it can be tricky to know how to respond.
Score: 1
"""

def build_score_prompt(instruction, response):
    return f"""{SYSTEM_PROMPT}

{FEW_SHOTS}

Now evaluate this pair.

Instruction: {instruction}
Response: {response}
Score:"""

# =========================
# 准备分数字符 token id
# 这里直接做“下一 token 分类”
# =========================
score_token_ids = {}
for s in ["1", "2", "3", "4", "5"]:
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f"Token ids for {s}: {ids}")
    if len(ids) != 1:
        raise ValueError(f"'{s}' is not a single token for this tokenizer, got {ids}")
    score_token_ids[int(s)] = ids[0]

candidate_scores = sorted(score_token_ids.keys())
candidate_token_ids = [score_token_ids[s] for s in candidate_scores]
@torch.no_grad()
def score_batch(batch_df):
    prompts = [
        build_score_prompt(row["instruction"], row["response"])
        for _, row in batch_df.iterrows()
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1400
    ).to(judge_model.device)

    outputs = judge_model(**inputs)
    next_token_logits = outputs.logits[:, -1, :]   # [batch, vocab]

    candidate_logits = next_token_logits[:, candidate_token_ids]  # [batch, 5]
    pred_idx = torch.argmax(candidate_logits, dim=-1)             # [batch]
    pred_scores = [candidate_scores[i] for i in pred_idx.tolist()]

    # 记录每个分数的相对“置信度”
    probs = torch.softmax(candidate_logits.float(), dim=-1)
    confs = torch.max(probs, dim=-1).values.tolist()

    result_rows = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        result = {
            "instruction": row["instruction"],
            "response": row["response"],
            "score": pred_scores[i],
            "score_confidence": confs[i],
        }
        if "original_instruction" in batch_df.columns:
            result["original_instruction"] = row["original_instruction"]
        result_rows.append(result)

    return result_rows
BATCH_SIZE = 32
results = []
print("\n开始评分...")
for start in range(0, len(df), BATCH_SIZE):
    batch_df = df.iloc[start:start+BATCH_SIZE].reset_index(drop=True)
    batch_results = score_batch(batch_df)
    results.extend(batch_results)
    print(f"评分进度: {min(start+BATCH_SIZE, len(df))}/{len(df)}")
scored_df = pd.DataFrame(results)
print("\n评分完成。")
print("\n分数分布：")
print(scored_df["score"].value_counts(dropna=False).sort_index())
high_df = scored_df[scored_df["score"] >= 4].reset_index(drop=True)
mid_df  = scored_df[scored_df["score"] == 3].reset_index(drop=True)
low_df  = scored_df[scored_df["score"] <= 2].reset_index(drop=True)

print("\n高质量样本数:", len(high_df))
print("中间样本数:", len(mid_df))
print("低质量样本数:", len(low_df))
curated_df = high_df[["instruction", "response"]].copy()

# =========================
# 给展示用的 10 条生成简短 reason
# =========================
DISPLAY_HIGH_N = min(5, len(high_df))
DISPLAY_LOW_N = min(5, len(low_df))

display_df = pd.concat([
    high_df.head(DISPLAY_HIGH_N),
    low_df.head(DISPLAY_LOW_N)
], axis=0).reset_index(drop=True)

REASON_PROMPT = """You are an evaluator.
Give one short sentence explaining why this instruction-response pair got the assigned score.

Instruction: {instruction}
Response: {response}
Score: {score}

Reason:"""

@torch.no_grad()
def generate_reason_batch(batch_df):
    prompts = [
        REASON_PROMPT.format(
            instruction=row["instruction"],
            response=row["response"],
            score=row["score"]
        )
        for _, row in batch_df.iterrows()
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1200
    ).to(judge_model.device)

    outputs = judge_model.generate(
        **inputs,
        max_new_tokens=24,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    reasons = []
    for i in range(len(prompts)):
        input_len = inputs["attention_mask"][i].sum().item()
        gen_ids = outputs[i][input_len:]
        reason = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        reasons.append(reason)
    return reasons

display_reasons = []
if len(display_df) > 0:
    for start in range(0, len(display_df), 8):
        batch_df = display_df.iloc[start:start+8].reset_index(drop=True)
        display_reasons.extend(generate_reason_batch(batch_df))

display_df["reason"] = display_reasons

print("\n================ HIGH QUALITY EXAMPLES ================\n")
for i in range(DISPLAY_HIGH_N):
    row = display_df.iloc[i]
    print(f"[High {i+1}]")
    print("Instruction:", row["instruction"])
    print("Response:", row["response"][:300], "...")
    print("Score:", row["score"], "| Confidence:", round(row["score_confidence"], 4))
    print("Reason:", row["reason"])
    print()

print("\n================ LOW QUALITY EXAMPLES ================\n")
for i in range(DISPLAY_HIGH_N, DISPLAY_HIGH_N + DISPLAY_LOW_N):
    row = display_df.iloc[i]
    print(f"[Low {i - DISPLAY_HIGH_N + 1}]")
    print("Instruction:", row["instruction"])
    print("Response:", row["response"][:300], "...")
    print("Score:", row["score"], "| Confidence:", round(row["score_confidence"], 4))
    print("Reason:", row["reason"])
    print()
scored_df.to_json(SCORED_OUTPUT_FILE, orient="records", lines=True)
curated_df.to_json(CURATED_OUTPUT_FILE, orient="records", lines=True)

print("\n已保存:")
print("-", SCORED_OUTPUT_FILE)
print("-", CURATED_OUTPUT_FILE)
!cp {SCORED_OUTPUT_FILE} /content/drive/MyDrive/
!cp {CURATED_OUTPUT_FILE} /content/drive/MyDrive/

原始样本数: 150
                                         instruction  \
0  advice-giver is trying to manipulate you, you ...   
1  What is the difference between using prop() an...   
2  I have a MySQL container running in Docker. I ...   

                                            response  \
0  When someone gives you unsolicited advice, it ...   
1  Always use the ```prop()``` method to enable o...   
2  By default after deployment MySQL has followin...   

                                original_instruction  
0              How to respond to unsolicited advice?  
1  I have to disable inputs at first and then on ...  
2  How to connect mysql workbench to running mysq...  

清洗后样本数: 103
                                         instruction  \
0  What is the difference between using prop() an...   
1  I have a MySQL container running in Docker. I ...   
2                              How do I clean slate?   

                                            response  \
0  Always use the ```prop

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Judge 模型加载完成。
padding_side = left
bf16 supported = True
Token ids for 1: [16]
Token ids for 2: [17]
Token ids for 3: [18]
Token ids for 4: [19]
Token ids for 5: [20]

开始评分...
评分进度: 32/103
评分进度: 64/103
评分进度: 96/103
评分进度: 103/103

评分完成。

分数分布：
score
1    31
2     8
3    16
4    37
5    11
Name: count, dtype: int64

高质量样本数: 48
中间样本数: 16
低质量样本数: 39

================ HIGH QUALITY EXAMPLES ================

[High 1]
Instruction: What is the difference between using prop() and attr() when enabling or disabling elements in jQuery?
Response: Always use the ```prop()``` method to enable or disable elements when using jQuery (see below for why).
In your case, it would be:
```$(&quot;#edit&quot;).click(function(event){
   event.preventDefault();
   $('.inputDisabled').prop(&quot;disabled&quot;, false); // Element(s) are now enabled.
});
`` ...
Score: 5 | Confidence: 0.7066
Reason: checked``` property of a checkbox. The ```.prop()``` method
should be used to set ```disabled``` and ```checked``` ins

In [3]:
import pandas as pd

scored_df = pd.read_json("self_curation_scored_v3.jsonl", lines=True)

curated_df = scored_df[scored_df["score"] >= 4][["instruction", "response"]].drop_duplicates().reset_index(drop=True)

print("最终用于第4步训练的样本数:", len(curated_df))

curated_df.to_json("self_curation_high_quality_final.jsonl", orient="records", lines=True)

最终用于第4步训练的样本数: 48


In [8]:

from datasets import Dataset
from huggingface_hub import login

login()

hf_dataset = Dataset.from_pandas(curated_df)

dataset_repo = "xinxinwanshishunli/self-curated-lima-qwen3"
hf_dataset.push_to_hub(dataset_repo)

print("已推送数据集到 HF:", dataset_repo)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmplleb9obj.parquet    : 100%|##########| 49.7kB / 49.7kB            

已推送数据集到 HF: xinxinwanshishunli/self-curated-lima-qwen3


In [4]:
# !pip install -U transformers datasets peft trl accelerate
import gc
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
MODEL_ID = "Qwen/Qwen3-1.7B"
INPUT_FILE = "self_curation_high_quality_final.jsonl"
OUTPUT_DIR = "./qwen3_forward_lora"
SAVE_PATH = "./qwen3_forward_lora_adapter"

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
df = pd.read_json(INPUT_FILE, lines=True)
df = df[["instruction", "response"]].dropna().drop_duplicates().reset_index(drop=True)

print("最终训练样本数:", len(df))
print(df.head(3))
dataset = Dataset.from_pandas(df)
split_ds = dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_ds["train"]
eval_data = split_ds["test"]

print("train size:", len(train_data))
print("eval size:", len(eval_data))

def format_forward(example):
    text = (
        f"Instruction:\n{example['instruction']}\n\n"
        f"Response:\n{example['response']}"
    )
    return {"text": text}

train_data = train_data.map(format_forward)
eval_data = eval_data.map(format_forward)

print("\n训练样例：")
print(train_data[0]["text"][:1000])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

try:
    del model, trainer
except:
    pass
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.use_cache = False
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=768,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    num_train_epochs=8,     # 数据很少，多跑几轮
    logging_steps=1,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"LoRA adapter 已保存到: {SAVE_PATH}")

CUDA available: True
Device: NVIDIA A100-SXM4-80GB
bf16 supported: True
最终训练样本数: 48
                                         instruction  \
0  What is the difference between using prop() an...   
1  What is the security concern of unlocking the ...   
2                 What is rooting an Android system?   

                                            response  
0  Always use the ```prop()``` method to enable o...  
1  It's a security concern. The Android documenta...  
2  In few words, rooting an Android system means ...  
train size: 43
eval size: 5


Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]


训练样例：
Instruction:
What is the implementation of std::function?

Response:
The implementation of ```std::function``` can differ from one implementation to another, but the core idea is that it uses type-erasure. While there are multiple ways of doing it, you can imagine a trivial (not optimal) solution could be like this (simplified for the specific case of ```std::function<int (double)>``` for the sake of simplicity):
```struct callable_base {
   virtual int operator()(double d) = 0;
   virtual ~callable_base() {}
};
template <typename F>
struct callable : callable_base {
   F functor;
   callable(F functor) : functor(functor) {}
   virtual int operator()(double d) { return functor(d); }
};
class function_int_double {
   std::unique_ptr<callable_base> c;
public:
   template <typename F>
   function(F f) {
      c.reset(new callable<F>(f));
   }
   int operator()(double d) { return c(d); }
// ...
};
```
In this simple approach the ```function``` object would store just a ```unique_ptr

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

Adding EOS to eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,2.588392,2.583703
2,2.877100,2.449872
3,2.500823,2.387082
4,2.641729,2.352750
5,2.726770,2.336026
6,2.179086,2.328571
7,1.950707,2.324738
8,2.051713,2.323397


LoRA adapter 已保存到: ./qwen3_forward_lora_adapter


In [13]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen3-1.7B"
ADAPTER_PATH = "./qwen3_forward_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

test_instructions = [
    "What is the difference between list.sort() and sorted() in Python?",
    "How can I learn to play the guitar?",
    "What is rooting an Android system?",
    "What is the security concern of unlocking the bootloader of an Android device?",
    "How can I improve sleep quality?"
]

for i, instr in enumerate(test_instructions, 1):
    prompt = f"Instruction:\n{instr}\n\nResponse:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=92,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    gen_ids = outputs[0][input_len:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    print("=" * 80)
    print(f"Example {i}")
    print("Instruction:", instr)
    print("Response:", response)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Example 1
Instruction: What is the difference between list.sort() and sorted() in Python?
Response: In this case, we will compare the two methods of sorting lists. The first method is to sort a list directly with the built-in .sort() function while the second one is to return an ordered version of it via the sorted() function.

When you call .sort(), any changes made on your original list are permanent. On the other hand, when using sorted(), the new variable that contains the result does not affect anything else as long as it's used
Example 2
Instruction: How can I learn to play the guitar?
Response: To start learning how to play an instrument like the guitar, you'll need some basic tools and materials. You’ll need a few things: strings for your guitar (you will have to choose from different types of string gauges), tuning pegs or tuners that allow you to tune them properly, fretboard markers which help with positioning notes on the neck of the guitar when playing along with other peo

In [6]:
# !pip install -U huggingface_hub

from huggingface_hub import login
login()

repo_name = "xinxinwanshishunli/qwen3-forward-lora-self-alignment"

trainer.model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("已推送到 HF:", repo_name)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|4         |  578kB / 12.9MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpvmlga50l/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

已推送到 HF: xinxinwanshishunli/qwen3-forward-lora-self-alignment
